# AI 실기 노트북 모의고사 3회 (모범 답안)

한 병원이 환자의 나이, 체질량지수(BMI), 혈압, 혈액 검사 수치 6개를 기록했습니다. 이 값으로 **1년 뒤의 질병 진행 지표**(숫자)를 예측하려고 합니다. 의학 지식은 필요하지 않습니다.

이 모의고사는 범주가 아니라 **수치를 예측하는 회귀**를 다룹니다. 신경망은 환자마다 예측값 **하나**를 출력합니다. 맞았는지가 아니라 **얼마나 차이 나는지(오차)** 로 평가합니다.

**총 8문항, 100점, 시험 120분**

- 공통 준비 셀을 먼저 실행하고, 각 문항 안의 셀은 위에서 아래로 실행하세요. 문항마다 필요한 자료를 따로 주므로 다른 문항의 답안은 필요하지 않습니다. 같은 문항의 (b), (c)는 앞 소문항의 결과를 써도 됩니다.
- `실행만 하세요` 셀은 고치지 않습니다. `작성하세요` 셀의 `None`이나 빈 곳을 코드로 바꾸세요. 중간 변수를 더하거나 여러 줄로 써도 됩니다. 정해진 변수 이름과 함수의 입력·반환 형식은 지키세요.
- 특정 정확도에 도달해야 하는 것은 아닙니다. 요구한 처리, 학습, 평가 과정을 수행하세요.
- 각 문항의 `확인` 셀을 실행하면 소문항별로 통과 여부와 점수를 보여 줍니다. 마지막 셀에서 총점을 확인하세요.
- 인터넷과 생성형 AI는 쓰지 않습니다. 필요한 데이터는 scikit-learn에 들어 있습니다.

## 데이터 안내

scikit-learn에 들어 있는 당뇨병(diabetes) 자료 442명의 **원래 값**을 섞어서 씁니다.

| 제공 자료 | 의미 |
|---|---|
| `dev` | 정답이 있는 개발용 데이터 362명. 1번에서 학습용과 검증용으로 나눕니다. |
| `new_samples` | 정답이 없는 새 환자 80명. |
| `patient_id` | 환자 번호(`P001` 형태). 모델 입력으로 쓰지 않습니다. |
| `target` | 예측할 1년 뒤 진행 지표. 대략 25 ~ 346 사이의 실수입니다. |
| `feature_cols` | 입력으로 쓸 열 10개: `age`, `sex`, `bmi`, `bp`, `s1` ~ `s6` |

`sex`는 1, 2로 기록된 범주이고, `s1` ~ `s6`은 혈액 검사 수치입니다. 결측치는 없습니다.

신경망 문항(5 ~ 7번)에서는 표준화한 입력과 **100으로 나눈 target**을 정답으로 씁니다. 숫자가 너무 크면 학습이 느려지기 때문입니다.

## 문항 구성

| 문항 | 내용 | 배점 |
|---|---|---|
| 1 | 정답 분포 확인하고 데이터 나누기 | 12점 |
| 2 | 정답과 관계가 큰 입력 찾기 | 10점 |
| 3 | 선형 회귀와 평균 예측 비교하기 | 12점 |
| 4 | 회귀 오차 지표 계산하기 | 14점 |
| 5 | 수치 하나를 예측하는 신경망 만들기 | 16점 |
| 6 | 학습 반복문 전체를 직접 쓰기 | 18점 |
| 7 | Dropout을 넣고 원래 단위로 예측하기 | 10점 |
| 8 | 오차가 큰 환자 찾기 | 8점 |

## 공통 준비

아래 두 셀은 고치지 말고 차례로 실행하세요. 첫 셀이 `practice_support.py`, `checker.py`를 준비합니다(Colab에서는 사이트에서 받아 옵니다).

In [ ]:
# 실행만 하세요 (준비 파일 받기)
# zip을 푼 폴더에서 열었으면 파일이 이미 있어 건너뜁니다. Colab에서는 사이트에서 받아 옵니다.
from pathlib import Path
import urllib.request

SUPPORT_BASE = 'https://unluckyboy214.github.io/exam-preparation-page/data/notebooks/aitest-nb-3/'
for name in ['practice_support.py', 'checker.py']:
    if not (Path.cwd() / name).is_file():
        urllib.request.urlretrieve(SUPPORT_BASE + name, name)
        print('받았어요:', name)
print('준비 파일 확인 끝')

In [ ]:
# 실행만 하세요
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display

if not (Path.cwd() / 'practice_support.py').is_file():
    raise FileNotFoundError('준비 파일이 없어요. 바로 위 셀을 먼저 실행하거나, zip 파일을 푼 폴더에서 이 노트북을 여세요.')
sys.path.insert(0, str(Path.cwd()))
from practice_support import (
    load_tables, ready_train_table, ready_raw_xy, ready_data, load_regression_example,
    make_practice_model, load_valid_predictions, seed_all,
)
import checker as check

torch.set_num_threads(2)
seed_all(3)
dev, new_samples, feature_cols = load_tables()
print('개발용 데이터:', len(dev), '/ 새 환자:', len(new_samples))
print('입력 열:', feature_cols)
display(dev.head(3))

## 1. 정답 분포 확인하고 데이터 나누기 [12점]

`dev`의 정답 열은 `target`입니다. 정답 값의 범위를 확인하고 80%를 학습용, 20%를 검증용으로 나눕니다.

회귀에서는 정답이 연속된 수치라서 `stratify`를 쓰지 않습니다.

In [ ]:
# 실행만 하세요
print('dev:', dev.shape)

**(a) [4점]** `dev['target']`의 평균, 표준편차, 최솟값, 최댓값을 구해 `target_stats`에 저장하세요. 인덱스가 `mean`, `std`, `min`, `max` 순서인 pandas Series입니다.

**(b) [5점]** `train_test_split`으로 `dev`를 나누어 `train_df`, `valid_df`에 저장하세요. `test_size=0.2`, `random_state=3`을 쓰고 `stratify`는 쓰지 않습니다.

**(c) [3점]** `train_df`의 `feature_cols` 열을 NumPy 배열 `X_tr`로, `target` 열을 NumPy 배열 `y_tr`로 만드세요.

In [ ]:
# (a) 정답의 요약 통계
target_stats = dev['target'].agg(['mean', 'std', 'min', 'max'])

# (b) 학습용과 검증용 나누기
train_df, valid_df = train_test_split(dev, test_size=0.2, random_state=3)

# (c) 입력 배열과 정답 배열
X_tr = train_df[feature_cols].to_numpy()
y_tr = train_df['target'].to_numpy()

**해설**

Series에 `agg([...])`를 쓰면 적은 이름 순서대로 인덱스가 붙은 Series가 나옵니다. `describe()`도 비슷하지만 `count`, 사분위수가 섞여 있고 순서가 다릅니다.

`stratify`는 범주의 비율을 지키는 기능이라 연속값에는 맞지 않습니다. 같은 `random_state`라도 `stratify`를 주면 나뉘는 결과가 달라집니다.

In [ ]:
# 실행만 하세요 (확인)
check.q1(globals())

## 2. 정답과 관계가 큰 입력 찾기 [10점]

`stats_df`는 따로 주는 **학습용의 원래 값**입니다. 1번 답안이 없어도 씁니다. 어떤 입력이 `target`과 함께 커지거나 작아지는지 상관계수로 확인합니다. 상관계수는 -1 ~ 1이고 1에 가까울수록 함께 커집니다.

In [ ]:
# 실행만 하세요
stats_df = ready_train_table()
print('학습용 환자 수:', len(stats_df))

**(a) [5점]** 입력 열 10개(`feature_cols`) 각각과 `target`의 상관계수를 구해 **큰 순서로** 정렬한 Series를 `corr_target`에 저장하세요. 인덱스는 입력 열 이름이고, `target` 자신은 넣지 않습니다.

**(b) [5점]** 주어진 `ax`에 가로축 `bmi`, 세로축 `target`인 산점도를 그리세요.

In [ ]:
# (a) 입력별 상관계수 (큰 순서)
corr_target = stats_df[feature_cols].corrwith(stats_df['target']).sort_values(ascending=False)

# (b) 산점도
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.scatter(stats_df['bmi'], stats_df['target'], s=12, alpha=0.7)

ax.set_xlabel('BMI')
ax.set_ylabel('Progress after 1 year')
plt.show()

**해설**

`corrwith`는 열마다 주어진 Series와의 상관계수를 구합니다. `stats_df[feature_cols + ['target']].corr()['target'].drop('target')`처럼 상관 행렬에서 `target` 열을 꺼내도 같은 값입니다.

`sort_values(ascending=False)`가 큰 순서입니다. 이 데이터에서는 `bmi`처럼 정답과 함께 커지는 입력이 위쪽에, 음의 상관이 있는 입력이 아래쪽에 옵니다.

In [ ]:
# 실행만 하세요 (확인)
check.q2(globals())

## 3. 선형 회귀와 평균 예측 비교하기 [12점]

모델이 쓸모 있는지 보려면 **아무것도 배우지 않은 기준**과 비교합니다. 여기서는 모든 환자를 학습용 평균으로 예측하는 것을 기준으로 삼습니다.

준비 셀이 표준화하지 않은 `X_train`, `X_valid`와 원래 단위의 정답 `y_train`, `y_valid`를 만듭니다. 1, 2번 답안은 필요하지 않습니다.

In [ ]:
# 실행만 하세요
raw = ready_raw_xy()
X_train, X_valid = raw['X_train'], raw['X_valid']
y_train, y_valid = raw['y_train'], raw['y_valid']
print('학습 입력:', X_train.shape, '/ 검증 입력:', X_valid.shape)

**(a) [6점]** `LinearRegression`으로 `lin`을 만들어 `X_train`, `y_train`으로 학습하고, `X_valid`의 예측을 `pred_lin`에 저장하세요.

**(b) [3점]** 검증용 환자 수만큼 **학습용 `y_train`의 평균**으로 채운 배열을 `pred_base`에 저장하세요.

**(c) [3점]** 두 예측의 MAE를 구해 `mae_lin`, `mae_base`에 저장하세요.

In [ ]:
# (a) 선형 회귀
lin = LinearRegression()
lin.fit(X_train, y_train)
pred_lin = lin.predict(X_valid)

# (b) 평균으로만 예측
pred_base = np.full(len(y_valid), y_train.mean())

# (c) 두 예측의 MAE
mae_lin = mean_absolute_error(y_valid, pred_lin)
mae_base = mean_absolute_error(y_valid, pred_base)

**해설**

기준 예측에는 **학습용** 평균을 씁니다. 검증용 평균을 쓰면 검증 정답을 미리 본 것이 됩니다.

`mae_lin`이 `mae_base`보다 작아야 모델이 평균보다 나은 것입니다. 선형 회귀는 열마다 계수를 따로 정하므로 표준화하지 않아도 예측값은 같습니다(계수의 크기만 달라집니다).

In [ ]:
# 실행만 하세요 (확인)
check.q3(globals())

## 4. 회귀 오차 지표 계산하기 [14점]

어떤 회귀 모델의 결과 40개를 줍니다. `actual`은 실제 값, `guessed`는 예측값이며 원래 단위입니다. 3번 답안은 쓰지 않습니다.

In [ ]:
# 실행만 하세요
actual, guessed = load_regression_example()
print('결과 수:', len(actual))
print('앞의 3개 실제 / 예측:', actual[:3], guessed[:3])

**(a) [4점]** 평균 절대 오차(MAE)를 `mae`에 저장하세요.

**(b) [5점]** 평균 제곱근 오차(RMSE)를 `rmse`에 저장하세요. MSE의 제곱근입니다.

**(c) [5점]** 결정 계수(R²)를 `r2`에 저장하세요.

In [ ]:
# (a) MAE
mae = mean_absolute_error(actual, guessed)

# (b) RMSE
rmse = np.sqrt(mean_squared_error(actual, guessed))

# (c) R²
r2 = r2_score(actual, guessed)

**해설**

MAE와 RMSE는 원래 값과 단위가 같고 작을수록 좋습니다. RMSE는 오차를 제곱해서 평균 내므로 큰 오차가 있으면 MAE보다 많이 커집니다. 최신 scikit-learn에는 `squared=False` 인자가 없으니 `np.sqrt`를 씁니다.

R²는 1에 가까울수록 좋고, 평균으로만 예측하면 0입니다. `r2_score`는 (실제, 예측) 순서가 바뀌면 값이 달라지므로 순서에 주의합니다.

In [ ]:
# 실행만 하세요 (확인)
check.q4(globals())

## 5. 수치 하나를 예측하는 신경망 만들기 [16점]

입력 10개를 받아 **예측값 하나**를 내는 신경망을 만듭니다. 구조는 `입력 10 → Linear(출력 32) → ReLU → Linear(출력 16) → ReLU → Linear(출력 1)`입니다. 회귀이므로 마지막에 활성화 함수를 넣지 않습니다.

준비 셀의 `yb`는 100으로 나눈 정답이며 모양이 `(B,)`입니다. 모델 출력 `(B, 1)`과 모양을 맞춰야 손실이 바르게 계산됩니다.

In [ ]:
# 실행만 하세요
seed_all(3)
d = ready_data()
xb = torch.tensor(d['X_train'][:8], dtype=torch.float32)
yb = torch.tensor(d['y_train'][:8], dtype=torch.float32)
print('입력 크기:', tuple(xb.shape), '/ 정답 크기:', tuple(yb.shape))

**(a) [7점]** `nn.Sequential`로 위 구조의 신경망을 만들어 `net`에 저장하세요.

**(b) [3점]** 평균 제곱 오차 손실 함수 `nn.MSELoss`를 만들어 `loss_fn`에 저장하세요.

**(c) [3점]** `net`의 모든 가중치를 갱신하는 `torch.optim.Adam`을 `optimizer`에 만드세요. 학습률은 0.01입니다.

**(d) [3점]** `yb`의 모양을 `(B, 1)`로 바꿔 `yb_col`에 저장하세요.

In [ ]:
# (a) 회귀 신경망
net = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)

# (b) 손실 함수
loss_fn = nn.MSELoss()

# (c) 가중치 갱신 방법
optimizer = torch.optim.Adam(net.parameters(), lr=0.01)

# (d) 정답 모양 맞추기
yb_col = yb.view(-1, 1)

# 아래는 확인용입니다. 고치지 마세요.
if net is not None and loss_fn is not None and yb_col is not None:
    print('출력 크기:', tuple(net(xb).shape), '/ 학습 전 손실:', float(loss_fn(net(xb), yb_col)))

**해설**

회귀의 마지막 층은 출력 1개이고 활성화 함수가 없습니다. ReLU를 끝에 두면 음수를 예측할 수 없게 됩니다.

`MSELoss`에 `(B, 1)`과 `(B,)`를 넣으면 브로드캐스팅으로 `(B, B)` 차이를 계산해 엉뚱한 손실이 나옵니다(경고만 나고 오류는 아닙니다). `yb.view(-1, 1)`이나 `yb.unsqueeze(1)`로 모양을 맞춥니다.

In [ ]:
# 실행만 하세요 (확인)
check.q5(globals())

## 6. 학습 반복문 전체를 직접 쓰기 [18점]

이번에는 학습 함수 없이 **반복문 전체**를 직접 씁니다. 준비 셀이 신경망 `train_net`(입력 10 → 은닉 16 → 예측 1), `loader`(최대 32개씩, 정답 모양 `(B, 1)`), 검증 데이터 `xv`, `yv`, 손실 함수 `train_loss_fn`, 옵티마이저 `train_opt`를 줍니다.

다시 학습하려면 **이 문항의 준비 셀부터** 다시 실행하세요. 학습 결과는 모범 답안과 같은지 비교하므로 아래 순서를 지키세요.

In [ ]:
# 실행만 하세요
seed_all(3)
d = ready_data()
train_set = TensorDataset(torch.tensor(d['X_train'], dtype=torch.float32),
                          torch.tensor(d['y_train'], dtype=torch.float32).view(-1, 1))
loader = DataLoader(train_set, batch_size=32, shuffle=True,
                    generator=torch.Generator().manual_seed(3))
xv = torch.tensor(d['X_valid'], dtype=torch.float32)
yv = torch.tensor(d['y_valid'], dtype=torch.float32).view(-1, 1)
train_net = make_practice_model()
train_loss_fn = nn.MSELoss()
train_opt = torch.optim.Adam(train_net.parameters(), lr=0.01)
print('학습 배치 수:', len(loader))

**(a) [12점]** 1부터 100까지 100 epoch를 학습하세요. 매 epoch마다 아래를 차례로 합니다.

1. `train_net`을 학습 모드로 바꾸고, `loader`의 배치마다 기울기 초기화 → 예측 → 손실 → 기울기 계산 → 가중치 갱신을 합니다.
2. 학습 손실 `tr_loss`는 **배치 손실 × 배치 크기를 모두 더해 전체 학습 데이터 수로 나눈 값**입니다.
3. 그다음 평가 모드로 바꾸고 기울기를 기록하지 않는 범위에서 `xv`, `yv`의 손실을 구해 `va_loss`(실수)에 저장합니다.
4. `epoch`, `train_loss`, `valid_loss` 세 키를 가진 딕셔너리를 `history`에 추가합니다.

**(b) [6점]** `history`를 DataFrame `history_df`로 만들고, 검증 손실이 가장 작은 epoch 번호(정수)를 `best_epoch`에 저장하세요.

In [ ]:
# (a) 학습 반복문
history = []
for epoch in range(1, 101):
    train_net.train()
    total, count = 0.0, 0
    for batch_x, batch_y in loader:
        train_opt.zero_grad()
        out = train_net(batch_x)
        loss = train_loss_fn(out, batch_y)
        loss.backward()
        train_opt.step()
        total += loss.item() * len(batch_y)
        count += len(batch_y)
    tr_loss = total / count

    train_net.eval()
    with torch.no_grad():
        va_loss = train_loss_fn(train_net(xv), yv).item()

    history.append({'epoch': epoch, 'train_loss': tr_loss, 'valid_loss': va_loss})

# (b) 기록 표와 가장 좋은 epoch
history_df = pd.DataFrame(history)
best_epoch = int(history_df.loc[history_df['valid_loss'].idxmin(), 'epoch'])

**해설**

마지막 배치는 32개보다 적을 수 있어서, 배치 손실을 그냥 평균 내지 않고 배치 크기로 가중합니다. `loss.item()`으로 실수로 바꿔 더해야 기울기 기록이 쌓이지 않습니다.

epoch마다 `train()`과 `eval()`을 오가야 합니다. `eval()`만 하고 다음 epoch에서 `train()`을 다시 부르지 않으면, Dropout이 있는 모델은 학습 중에도 Dropout이 꺼진 채로 남습니다.

In [ ]:
# 실행만 하세요 (확인)
check.q6(globals())

## 7. Dropout을 넣고 원래 단위로 예측하기 [10점]

새 모델은 `입력 10 → Linear(출력 32) → ReLU → Dropout → Linear(출력 1)`입니다. 이 모델은 100으로 나눈 값을 예측하도록 만들어졌으므로, 사람이 볼 때는 **100을 곱해 원래 단위**로 돌립니다. 이 문항에서는 학습하지 않습니다.

`sample_x`는 `(5, 10)` 텐서입니다.

In [ ]:
# 실행만 하세요
seed_all(17)
sample_x = torch.tensor(ready_data()['X_valid'][:5], dtype=torch.float32)

**(a) [6점]** `nn.Sequential`로 위 구조의 모델을 만들어 `drop_net`에 저장하세요. `nn.Dropout`의 `p`는 0.1이고 ReLU 바로 뒤에 둡니다.

**(b) [4점]** `predict_values(model, x)`를 완성하세요. 평가 모드로 바꾸고 기울기를 기록하지 않는 범위에서 예측한 뒤, 100을 곱해 모양 `(샘플 수,)`인 **NumPy 배열**을 `out`에 저장합니다.

In [ ]:
# (a) Dropout이 있는 회귀 신경망
drop_net = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Dropout(p=0.1),
    nn.Linear(32, 1),
)

# (b) 원래 단위로 예측하는 함수
def predict_values(model, x):
    out = None
    model.eval()
    with torch.no_grad():
        out = model(x).squeeze(1).numpy() * 100
    return out

if drop_net is not None:
    print('예측값:', predict_values(drop_net, sample_x))

**해설**

`squeeze(1)`로 `(B, 1)`을 `(B,)`로 만들고 `.numpy()`로 NumPy 배열로 바꿉니다. `no_grad()` 안에서 계산한 텐서는 기울기 기록이 없어 바로 `.numpy()`를 쓸 수 있습니다. 밖에서 계산했다면 `.detach().numpy()`가 필요합니다.

정답을 100으로 나누어 학습했으므로 예측에도 100을 곱해야 원래 단위가 됩니다.

In [ ]:
# 실행만 하세요 (확인)
check.q7(globals())

## 8. 오차가 큰 환자 찾기 [8점]

기준 모델이 검증용 환자에 대해 낸 예측 `pred`와 실제 값 `actual`, 같은 순서의 환자 번호 `ids`(NumPy 배열)를 줍니다. 모두 원래 단위입니다. 오차가 큰 환자를 골라 기록을 다시 확인하려고 합니다.

In [ ]:
# 실행만 하세요
ids, actual, pred = load_valid_predictions()
print('검증용 환자 수:', len(ids))

**(a) [4점]** 환자마다 `|실제 - 예측|`을 구해 NumPy 배열 `abs_err`에 저장하세요.

**(b) [4점]** `abs_err`가 큰 순서대로 5명의 환자 번호를 `top5_ids`에 저장하세요. 길이 5의 NumPy 배열이며 첫 번째가 오차가 가장 큰 환자입니다.

In [ ]:
# (a) 환자별 절대 오차
abs_err = np.abs(actual - pred)

# (b) 오차가 큰 5명
order = np.argsort(abs_err)[::-1]
top5_ids = ids[order[:5]]

**해설**

`np.argsort`는 작은 값부터의 **위치**를 돌려줍니다. `[::-1]`로 뒤집거나 `np.argsort(-abs_err)`를 쓰면 큰 순서가 됩니다. 그 위치로 `ids`를 고르면 환자 번호가 나옵니다.

절댓값을 빼먹으면 예측이 실제보다 큰 경우(음수 오차)가 뒤로 밀려 잘못 고르게 됩니다.

In [ ]:
# 실행만 하세요 (확인)
check.q8(globals())

## 총점

확인 셀을 실행한 문항의 점수를 모읍니다(확인하지 않은 문항은 0점). 나오는 **사이트에 기록하기** 링크를 누르면 사이트의 "내 기록"에 점수가 저장됩니다. 링크를 연 브라우저에 저장됩니다.

In [ ]:
# 실행만 하세요 (총점과 "사이트에 기록하기" 링크)
check.total(points=[12, 10, 12, 14, 16, 18, 10, 8],
            record_url='https://unluckyboy214.github.io/exam-preparation-page/pages/my.html?nb=aitest-nb-3&v=1')